# 04 — AI Model Performance & Customer Usage Analytics

**Goal:** Consolidate 20M+ daily AI inference logs via Spark ETL into Snowflake. Enhance customer satisfaction by 18% through usage trend analysis.  
**Stack (local simulation):** Python · Pandas · Parquet  
**Production stack:** AWS · Spark · Snowflake · Power BI

---

In [ ]:
import os, sys, subprocess, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.dpi'] = 120

RAW_DIR  = os.path.join('..', 'data', 'raw')
PROC_DIR = os.path.join('..', 'data', 'processed')

print('Ready ✅')

## Step 1 — Generate Synthetic Inference Logs

In [ ]:
result = subprocess.run(['python', os.path.join('..', 'scripts', 'generate_data.py')],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 2 — Run Spark ETL Pipeline

In [ ]:
result = subprocess.run(['python', os.path.join('..', 'scripts', 'spark_etl_pipeline.py')],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 3 — Run Usage Trend Analysis

In [ ]:
result = subprocess.run(['python', os.path.join('..', 'scripts', 'usage_trend_analysis.py')],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 4 — Load & Visualise Results

In [ ]:
daily     = pd.read_parquet(os.path.join(PROC_DIR, 'fact_daily_model_metrics.parquet'))
customers = pd.read_parquet(os.path.join(PROC_DIR, 'dim_customer_usage.parquet'))
usecase   = pd.read_parquet(os.path.join(PROC_DIR, 'fact_usecase_performance.parquet'))
sim       = pd.read_parquet(os.path.join(PROC_DIR, 'csat_intervention_simulation.parquet'))

print(f'Daily metrics rows : {len(daily):,}')
print(f'Customer rows      : {len(customers):,}')
print(f'Use-case rows      : {len(usecase):,}')

In [ ]:
# ── Chart 1: Model Performance Leaderboard ────────────────────────────────────
leaderboard = (
    daily.groupby('model')
    .agg(avg_csat=('avg_csat','mean'), avg_latency=('avg_latency_ms','mean'),
         total_cost=('total_cost_usd','sum'), total_calls=('total_calls','sum'))
    .sort_values('avg_csat', ascending=False)
    .round(3)
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].barh(leaderboard.index, leaderboard['avg_csat'],
             color=sns.color_palette('husl', len(leaderboard)))
axes[0].set_title('Average CSAT Score', fontweight='bold')
axes[0].set_xlabel('CSAT (1–5)')

axes[1].barh(leaderboard.index, leaderboard['avg_latency'],
             color=sns.color_palette('husl', len(leaderboard)))
axes[1].set_title('Average Latency (ms)', fontweight='bold')
axes[1].set_xlabel('Latency (ms)')

axes[2].barh(leaderboard.index, leaderboard['total_cost'] / 1000,
             color=sns.color_palette('husl', len(leaderboard)))
axes[2].set_title('Total Cost (USD thousands)', fontweight='bold')
axes[2].set_xlabel('Cost ($K)')

plt.suptitle('AI Model Performance Leaderboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 2: CSAT Trend Over Time ─────────────────────────────────────────────
daily['date']  = pd.to_datetime(daily['date'])
daily['month'] = daily['date'].dt.to_period('M').dt.to_timestamp()

csat_trend = (
    daily.groupby(['month', 'model'])
    .apply(lambda g: np.average(g['avg_csat'], weights=g['total_calls']))
    .rename('weighted_csat')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
for model, grp in csat_trend.groupby('model'):
    ax.plot(grp['month'], grp['weighted_csat'], marker='o', ms=3, label=model)
ax.set_title('CSAT Score Trend by Model Over Time', fontsize=13, fontweight='bold')
ax.set_ylabel('Weighted CSAT (1–5)')
ax.set_xlabel('Month')
ax.legend(loc='lower right')
ax.set_ylim(1, 5.5)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 3: CSAT by Use Case ─────────────────────────────────────────────────
csat_uc = (
    usecase.groupby('use_case')
    .apply(lambda g: np.average(g['avg_csat'], weights=g['calls']))
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(csat_uc.index, csat_uc.values,
               color=sns.color_palette('husl', len(csat_uc)))
ax.bar_label(bars, fmt='%.2f', padding=4)
ax.set_title('Average CSAT by Use Case', fontsize=13, fontweight='bold')
ax.set_xlabel('CSAT Score')
ax.set_xlim(0, 6)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 4: 18% CSAT Improvement Simulation ─────────────────────────────────
baseline_dist = sim['baseline_csat'].clip(1, 5)
improved_dist = sim['improved_csat'].clip(1, 5)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(baseline_dist, bins=50, alpha=0.6, label=f'Baseline (μ={baseline_dist.mean():.2f})',
        color='#e74c3c', density=True)
ax.hist(improved_dist, bins=50, alpha=0.6, label=f'Post-Intervention (μ={improved_dist.mean():.2f})',
        color='#27ae60', density=True)
ax.axvline(baseline_dist.mean(), color='#e74c3c', linestyle='--', linewidth=1.5)
ax.axvline(improved_dist.mean(), color='#27ae60', linestyle='--', linewidth=1.5)

lift = (improved_dist.mean() - baseline_dist.mean()) / baseline_dist.mean() * 100
ax.set_title(f'CSAT Distribution: Baseline vs Post-Intervention (+{lift:.1f}% lift)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('CSAT Score')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 5: Daily Inference Volume ──────────────────────────────────────────
daily_vol = daily.groupby('date')['total_calls'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(daily_vol['date'], daily_vol['total_calls'] / 1000, alpha=0.4, color='#3498db')
ax.plot(daily_vol['date'], daily_vol['total_calls'] / 1000, color='#3498db', linewidth=1.5)
ax.set_title('Daily AI Inference Volume', fontsize=13, fontweight='bold')
ax.set_ylabel('API Calls (thousands)')
ax.set_xlabel('Date')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}K'))
plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| Daily inference logs (production scale) | **20M+** |
| Synthetic records generated | 2M |
| Models benchmarked | 5 |
| CSAT improvement (simulated) | **+18%** |
| Snowflake tables produced | 4 |
| Pipeline: Extract → Transform → Load | Spark ETL |

**Key interventions for CSAT gain:**
- Route latency-sensitive use cases to faster models (llama-3, mistral)
- Implement automatic fallback routing on error
- Proactive outreach for customers with CSAT < 3
- Weekly model performance reports to account teams